<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 11: Filtering and Sorting Rows

In [ ]:
%%bash

pip install --upgrade polars

In [ ]:
import polars as pl
pl.__version__

In [ ]:
%%bash

curl -L -o "sample_data/tools.csv" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/tools.csv"

In [ ]:
tools = pl.read_csv("sample_data/tools.csv")
tools

## Filtering Rows

### Filtering Based on Expressions

In [ ]:
tools.filter(pl.col("cordless") & (pl.col("brand") == "Makita"))

In [ ]:
tools.filter(pl.col("cordless"), pl.col("brand") == "Makita")

### Filtering Based on Column Names

In [ ]:
tools.filter("cordless")

### Filtering Based on Constraints

In [ ]:
tools.filter(cordless=True, brand="Makita")

## Sorting Rows

### Sorting Based on a Single Column

In [ ]:
tools.sort("price")

### Sorting in Reverse

In [ ]:
tools.sort("price", descending=True)

In [ ]:
# This raises a TypeError:
tools.sort("price", ascending=False)

### Sorting Based on Multiple Columns

In [ ]:
tools.sort("brand", "price")

In [ ]:
tools.sort("brand", "price", descending=[False, True])

### Sorting Based on Expressions

In [ ]:
tools.sort(pl.col("rpm") / pl.col("price"))

### Sorting Nested Data Types

In [ ]:
lists = pl.DataFrame({"lists": [[2, 2], [2, 1, 3], [1]]})
lists.sort("lists")

In [ ]:
structs = pl.DataFrame(
    {
        "structs": [
            {"a": 1, "b": 2, "c": 3},
            {"a": 1, "b": 3, "c": 1},
            {"a": 1, "b": 1, "c": 2},
        ]
    }
)
structs.sort("structs")

In [ ]:
tools_collection = tools.group_by("brand").agg(collection=pl.struct(pl.all()))
tools_collection

In [ ]:
tools_collection.sort(pl.col("collection").list.len(), descending=True)

In [ ]:
tools_collection.sort(
    pl.col("collection")
    .list.eval(pl.element().struct.field("price"))
    .list.mean()
)

In [ ]:
tools_collection.with_columns(
    mean_price=pl.col("collection")
    .list.eval(pl.element().struct.field("price"))
    .list.mean()
).sort("mean_price")

## Related Row Operations

### Filtering Missing Values

In [ ]:
tools.drop_nulls("rpm").height

In [ ]:
tools.filter(pl.all_horizontal(pl.all().is_not_null())).height

### Slicing

In [ ]:
tools.with_row_index().gather_every(2).head(3)

### Top and Bottom

In [ ]:
tools.top_k(3, by="price")

### Sampling

In [ ]:
tools.sample(fraction=0.2)

### Semi-Joins

In [ ]:
saws = pl.DataFrame(
    {
        "tool": [
            "Table Saw",
            "Plunge Cut Saw",
            "Miter Saw",
            "Jigsaw",
            "Bandsaw",
            "Chainsaw",
            "Seesaw",
        ]
    }
)
tools.join(saws, how="semi", on="tool")